# Week 4

In [1]:
import pandas as pd
from pathlib import Path

In [40]:
sold_with_mortgage_rates = pd.read_csv("/Users/amyliu/Desktop/IDX/data/generated/sold_with_mortgage_rates.csv")
listings_with_mortgage_rates = pd.read_csv("/Users/amyliu/Desktop/IDX/data/generated/listing_with_mortgage_rates.csv")

/var/folders/23/ffpfhtdx4wx901rhxgr55x4w0000gn/T/ipykernel_22557/1683297970.py:1: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: WaterfrontYN, 3: ListAgentEmail, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  sold_with_mortgage_rates = pd.read_csv("/Users/amyliu/Desktop/IDX/data/generated/sold_with_mortgage_rates.csv")
/var/folders/23/ffpfhtdx4wx901rhxgr55x4w0000gn/T/ipykernel_22557/1683297970.py:2: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  listings_with_mortgage_rates = pd.read_csv("/Users/amyliu/Desktop/IDX/data/generated/listing_with_mortgage_rates.csv")


In [3]:
sold_with_rates = sold_with_mortgage_rates.copy()
listing_with_rates = listings_with_mortgage_rates.copy()

### Step 1: Convert date fields

For sold and listings, the key fields are:

* CloseDate
* PurchaseContractDate
* ListingContractDate
* ContractStatusChangeDate

In [4]:
sold_date_cols = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']

for col in sold_date_cols:
    if col in sold_with_rates.columns:
        sold_with_rates[col] = pd.to_datetime(sold_with_rates[col], errors='coerce')

In [5]:
listing_date_cols = ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']

for col in listing_date_cols:
    if col in listing_with_rates.columns:
        listing_with_rates[col] = pd.to_datetime(listing_with_rates[col], errors='coerce')

### Step 2: Remove redundant / unnecessary columns

#### 1. Drop non-core columns with >90% missing

**Core fields in Sold dataset**

The core fields in the sold dataset are the variables most important for residential market analysis.\
These include price-related fields such as `ClosePrice`, `ListPrice`, and `OriginalListPrice`; \
date fields such as `CloseDate`, `ListingContractDate`,`PurchaseContractDate`, and `ContractStatusChangeDate`; \
property-characteristic fields such as `LivingArea`, `LotSizeAcres`, `LotSizeSquareFeet`, `BedroomsTotal`, `BathroomsTotalInteger`, `DaysOnMarket`, `YearBuilt`, and `PropertySubType`;\
geographic fields such as `CountyOrParish`, `City`, `PostalCode`, `Latitude`, `Longitude`, and `MLSAreaMajor`;\
and competition-related or tracking fields such as `ListOfficeName`, `BuyerOfficeName`, `PropertyType`, `ListingKey`, `ListingKeyNumeric`, `ListingId`, `year_month`, and `rate_30yr_fixed`.


In [6]:
core_fields_sold = ['ClosePrice', 'ListPrice', 'OriginalListPrice','CloseDate', 'ListingContractDate', 'PurchaseContractDate', 'ContractStatusChangeDate','LivingArea', 'LotSizeAcres', 'LotSizeSquareFeet','BedroomsTotal', 'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt','CountyOrParish', 'City', 'PostalCode', 'Latitude', 'Longitude','PropertyType', 'PropertySubType', 'MLSAreaMajor','ListOfficeName', 'BuyerOfficeName','year_month', 'rate_30yr_fixed']

In [7]:
high_missing_cols_sold = (sold_with_rates.isnull().mean().loc[lambda x: x > 0.90].index.tolist())

In [8]:
high_missing_cols_sold

['WaterfrontYN',
 'BasementYN',
 'FireplacesTotal',
 'AboveGradeFinishedArea',
 'TaxAnnualAmount',
 'BuilderName',
 'TaxYear',
 'ElementarySchoolDistrict',
 'CoBuyerAgentFirstName',
 'BelowGradeFinishedArea',
 'BusinessType',
 'CoveredSpaces',
 'LotSizeDimensions',
 'MiddleOrJuniorSchoolDistrict']

In [9]:
drop_cols_sold = [col for col in high_missing_cols_sold if col not in core_fields_sold]
sold_with_rates = sold_with_rates.drop(columns=drop_cols_sold)

print("Columns dropped from sold dataset:")
print(drop_cols_sold)
print("New sold dataset shape:", sold_with_rates.shape)

Columns dropped from sold dataset:
['WaterfrontYN', 'BasementYN', 'FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'BuilderName', 'TaxYear', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'LotSizeDimensions', 'MiddleOrJuniorSchoolDistrict']
New sold dataset shape: (591733, 72)


**Core fields in Listing dataset**

The core fields in the listing dataset are the variables most important for listing-side residential market analysis. 
These include price-related fields such as `ListPrice`, `OriginalListPrice`, and `ClosePrice`; \
date fields such as `ListingContractDate`, `ContractStatusChangeDate`, `PurchaseContractDate`, and `CloseDate`; \
property-characteristic fields such as `LivingArea`, `LotSizeAcres`, `LotSizeSquareFeet`, `BedroomsTotal`, `BathroomsTotalInteger`, `DaysOnMarket`, `YearBuilt`, and `PropertySubType`; \
geographic fields such as `CountyOrParish`, `City`, `PostalCode`, `Latitude`, `Longitude`, and `MLSAreaMajor`; \
and listing-related or tracking fields such as `ListOfficeName`, `PropertyType`, `ListingKey`, `ListingKeyNumeric`, `ListingId`, `year_month`, and `rate_30yr_fixed`. \
These core fields were retained even if partially missing because they are required for pricing, timing, geographic, and property-level analysis.

In [10]:
core_fields_listing = ['ClosePrice', 'ListPrice', 'OriginalListPrice','CloseDate', 'ListingContractDate', 'PurchaseContractDate', 'ContractStatusChangeDate','LivingArea', 'LotSizeAcres', 'LotSizeSquareFeet','BedroomsTotal', 'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt','CountyOrParish', 'City', 'PostalCode', 'Latitude', 'Longitude','PropertyType', 'PropertySubType', 'MLSAreaMajor','ListOfficeName','year_month', 'rate_30yr_fixed']

In [11]:
high_missing_cols_listing = (listing_with_rates.isnull().mean().loc[lambda x: x > 0.90].index.tolist())
high_missing_cols_listing

['FireplacesTotal',
 'AboveGradeFinishedArea',
 'TaxAnnualAmount',
 'ElementarySchool',
 'BuilderName',
 'TaxYear',
 'ElementarySchoolDistrict',
 'CoBuyerAgentFirstName',
 'BelowGradeFinishedArea',
 'BusinessType',
 'CoveredSpaces',
 'MiddleOrJuniorSchool',
 'LotSizeDimensions',
 'MiddleOrJuniorSchoolDistrict']

In [12]:
drop_cols_listing = [col for col in high_missing_cols_listing if col not in core_fields_listing]
listing_with_rates = listing_with_rates.drop(columns=drop_cols_listing)

print("Columns dropped from listing dataset:")
print(drop_cols_listing)
print("New listing dataset shape:", listing_with_rates.shape)

Columns dropped from listing dataset:
['FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount', 'ElementarySchool', 'BuilderName', 'TaxYear', 'ElementarySchoolDistrict', 'CoBuyerAgentFirstName', 'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces', 'MiddleOrJuniorSchool', 'LotSizeDimensions', 'MiddleOrJuniorSchoolDistrict']
New listing dataset shape: (852963, 72)


Columns with more than 90% missing values were reviewed for analytical relevance. Non-core variables with extremely limited coverage were dropped from the working dataset, while core fields related to pricing, dates, geography, property characteristics, and mortgage enrichment were retained.

#### 2. Drop obvious duplicate .1 columns in listings if they are true duplicates

In [15]:
duplicate_pairs = [
    ('PropertyType', 'PropertyType.1'),
    ('ListAgentFirstName', 'ListAgentFirstName.1'),
    ('DaysOnMarket', 'DaysOnMarket.1'),
    ('LivingArea', 'LivingArea.1'),
    ('Longitude', 'Longitude.1'),
    ('Latitude', 'Latitude.1'),
    ('ListPrice', 'ListPrice.1'),
    ('ListAgentLastName', 'ListAgentLastName.1'),
    ('CloseDate', 'CloseDate.1'),
    ('BuyerOfficeName', 'BuyerOfficeName.1'),
    ('UnparsedAddress', 'UnparsedAddress.1')
]

for col1, col2 in duplicate_pairs:
    same = listing_with_rates[col1].equals(listing_with_rates[col2])
    print(f"{col1} vs {col2}: {same}")

PropertyType vs PropertyType.1: True
ListAgentFirstName vs ListAgentFirstName.1: True
DaysOnMarket vs DaysOnMarket.1: True
LivingArea vs LivingArea.1: True
Longitude vs Longitude.1: True
Latitude vs Latitude.1: True
ListPrice vs ListPrice.1: True
ListAgentLastName vs ListAgentLastName.1: True
CloseDate vs CloseDate.1: False
BuyerOfficeName vs BuyerOfficeName.1: True
UnparsedAddress vs UnparsedAddress.1: True


In [17]:
true_duplicate_cols = []
for col1, col2 in duplicate_pairs:
    if listing_with_rates[col1].equals(listing_with_rates[col2]):
        true_duplicate_cols.append(col2)

print("True duplicate .1 columns to drop:")
print(true_duplicate_cols)

True duplicate .1 columns to drop:
['PropertyType.1', 'ListAgentFirstName.1', 'DaysOnMarket.1', 'LivingArea.1', 'Longitude.1', 'Latitude.1', 'ListPrice.1', 'ListAgentLastName.1', 'BuyerOfficeName.1', 'UnparsedAddress.1']


In [18]:
listing_with_rates = listing_with_rates.drop(columns=true_duplicate_cols)

### Step 3: Confirm numeric fields are numeric

In [19]:
sold_numeric_cols = [
    'ClosePrice', 'ListPrice', 'OriginalListPrice',
    'LivingArea', 'LotSizeAcres', 'LotSizeSquareFeet',
    'BedroomsTotal', 'BathroomsTotalInteger', 'DaysOnMarket',
    'YearBuilt', 'Latitude', 'Longitude',
    'rate_30yr_fixed'
]

sold_with_rates[sold_numeric_cols].dtypes

ClosePrice               float64
ListPrice                float64
OriginalListPrice        float64
LivingArea               float64
LotSizeAcres             float64
LotSizeSquareFeet        float64
BedroomsTotal            float64
BathroomsTotalInteger    float64
DaysOnMarket               int64
YearBuilt                float64
Latitude                 float64
Longitude                float64
rate_30yr_fixed          float64
dtype: object

In [20]:
listing_numeric_cols = [
    'ClosePrice', 'ListPrice', 'OriginalListPrice',
    'LivingArea', 'LotSizeAcres', 'LotSizeSquareFeet',
    'BedroomsTotal', 'BathroomsTotalInteger', 'DaysOnMarket',
    'YearBuilt', 'Latitude', 'Longitude',
    'rate_30yr_fixed'
]

print("Listing dataset numeric field types:")
print(listing_with_rates[listing_numeric_cols].dtypes)

Listing dataset numeric field types:
ClosePrice               float64
ListPrice                float64
OriginalListPrice        float64
LivingArea               float64
LotSizeAcres             float64
LotSizeSquareFeet        float64
BedroomsTotal            float64
BathroomsTotalInteger    float64
DaysOnMarket               int64
YearBuilt                float64
Latitude                 float64
Longitude                float64
rate_30yr_fixed          float64
dtype: object


### Step 4: Create invalid-value flags

In [21]:
sold_with_rates['invalid_closeprice_flag'] = sold_with_rates['ClosePrice'] <= 0
sold_with_rates['invalid_livingarea_flag'] = sold_with_rates['LivingArea'] <= 0
sold_with_rates['invalid_dom_flag'] = sold_with_rates['DaysOnMarket'] < 0
sold_with_rates['invalid_bedrooms_flag'] = sold_with_rates['BedroomsTotal'] < 0
sold_with_rates['invalid_bathrooms_flag'] = sold_with_rates['BathroomsTotalInteger'] < 0

In [22]:
sold_invalid_flag_counts = {
    'invalid_closeprice_flag': sold_with_rates['invalid_closeprice_flag'].sum(),
    'invalid_livingarea_flag': sold_with_rates['invalid_livingarea_flag'].sum(),
    'invalid_dom_flag': sold_with_rates['invalid_dom_flag'].sum(),
    'invalid_bedrooms_flag': sold_with_rates['invalid_bedrooms_flag'].sum(),
    'invalid_bathrooms_flag': sold_with_rates['invalid_bathrooms_flag'].sum()
}

sold_invalid_flag_counts

{'invalid_closeprice_flag': np.int64(35),
 'invalid_livingarea_flag': np.int64(485),
 'invalid_dom_flag': np.int64(60),
 'invalid_bedrooms_flag': np.int64(0),
 'invalid_bathrooms_flag': np.int64(0)}

In [23]:
listing_with_rates['invalid_closeprice_flag'] = listing_with_rates['ClosePrice'] <= 0
listing_with_rates['invalid_livingarea_flag'] = listing_with_rates['LivingArea'] <= 0
listing_with_rates['invalid_dom_flag'] = listing_with_rates['DaysOnMarket'] < 0
listing_with_rates['invalid_bedrooms_flag'] = listing_with_rates['BedroomsTotal'] < 0
listing_with_rates['invalid_bathrooms_flag'] = listing_with_rates['BathroomsTotalInteger'] < 0

### Step 5: Create date consistency flags

In [24]:
date_cols = ['ListingContractDate', 'PurchaseContractDate', 'CloseDate']

for col in date_cols:
    if col in sold_with_rates.columns:
        sold_with_rates[col] = pd.to_datetime(sold_with_rates[col], errors='coerce')

In [25]:
sold_with_rates['listing_after_close_flag'] = sold_with_rates['ListingContractDate'] > sold_with_rates['CloseDate']
sold_with_rates['purchase_after_close_flag'] = sold_with_rates['PurchaseContractDate'] > sold_with_rates['CloseDate']
sold_with_rates['negative_timeline_flag'] = sold_with_rates['PurchaseContractDate'] < sold_with_rates['ListingContractDate']

In [26]:
print("Listing after close:", sold_with_rates['listing_after_close_flag'].sum())
print("Purchase after close:", sold_with_rates['purchase_after_close_flag'].sum())
print("Negative timeline:", sold_with_rates['negative_timeline_flag'].sum())

Listing after close: 109
Purchase after close: 356
Negative timeline: 379


In [27]:
date_cols = ['ListingContractDate', 'PurchaseContractDate', 'CloseDate']

for col in date_cols:
    if col in listing_with_rates.columns:
        listing_with_rates[col] = pd.to_datetime(listing_with_rates[col], errors='coerce')

listing_with_rates['listing_after_close_flag'] = listing_with_rates['ListingContractDate'] > listing_with_rates['CloseDate']
listing_with_rates['purchase_after_close_flag'] = listing_with_rates['PurchaseContractDate'] > listing_with_rates['CloseDate']
listing_with_rates['negative_timeline_flag'] = listing_with_rates['PurchaseContractDate'] < listing_with_rates['ListingContractDate']

print("Listing after close:", listing_with_rates['listing_after_close_flag'].sum())
print("Purchase after close:", listing_with_rates['purchase_after_close_flag'].sum())
print("Negative timeline:", listing_with_rates['negative_timeline_flag'].sum())

Listing after close: 135
Purchase after close: 351
Negative timeline: 397


In [28]:
listing_invalid_flag_counts = {
    'invalid_closeprice_flag': listing_with_rates['invalid_closeprice_flag'].sum(),
    'invalid_livingarea_flag': listing_with_rates['invalid_livingarea_flag'].sum(),
    'invalid_dom_flag': listing_with_rates['invalid_dom_flag'].sum(),
    'invalid_bedrooms_flag': listing_with_rates['invalid_bedrooms_flag'].sum(),
    'invalid_bathrooms_flag': listing_with_rates['invalid_bathrooms_flag'].sum()
}

listing_invalid_flag_counts

{'invalid_closeprice_flag': np.int64(12),
 'invalid_livingarea_flag': np.int64(842),
 'invalid_dom_flag': np.int64(37),
 'invalid_bedrooms_flag': np.int64(0),
 'invalid_bathrooms_flag': np.int64(0)}

### Step 6: Create geographic data quality flags

Flag:

* missing coordinates
* Latitude = 0 or Longitude = 0
* Longitude > 0
* out-of-state or implausible coordinates 

In [29]:
sold_with_rates['Latitude'] = pd.to_numeric(sold_with_rates['Latitude'], errors='coerce')
sold_with_rates['Longitude'] = pd.to_numeric(sold_with_rates['Longitude'], errors='coerce')

In [47]:
sold_with_rates['missing_coord_flag'] = sold_with_rates['Latitude'].isna() | sold_with_rates['Longitude'].isna()

sold_with_rates['zero_coord_flag'] = (
    (sold_with_rates['Latitude'] == 0) |
    (sold_with_rates['Longitude'] == 0)
)

sold_with_rates['positive_longitude_flag'] = sold_with_rates['Longitude'] > 0
#California coordinates should be negative
sold_with_rates['out_of_state_flag'] = (
    (sold_with_rates['Latitude'] < 30) | (sold_with_rates['Latitude'] > 42.5) |
    (sold_with_rates['Longitude'] < -124.5) | (sold_with_rates['Longitude'] > -114)
)

sold_with_rates['implausible_coord_flag'] = (
    (sold_with_rates['Latitude'] < -90) | (sold_with_rates['Latitude'] > 90) |
    (sold_with_rates['Longitude'] < -180) | (sold_with_rates['Longitude'] > 180)
)

In [45]:
print("Missing coordinates:", sold_with_rates['missing_coord_flag'].sum())
print("Zero coordinates:", sold_with_rates['zero_coord_flag'].sum())
print("Out of state coordinates:", sold_with_rates['out_of_state_flag'].sum())
print("Positive longitude:", sold_with_rates['positive_longitude_flag'].sum())
print("Implausible coordinates:", sold_with_rates['implausible_coord_flag'].sum())

Missing coordinates: 19030
Zero coordinates: 43
Out of state coordinates: 177
Positive longitude: 62
Implausible coordinates: 4


In [48]:
listing_with_rates['Latitude'] = pd.to_numeric(listing_with_rates['Latitude'], errors='coerce')
listing_with_rates['Longitude'] = pd.to_numeric(listing_with_rates['Longitude'], errors='coerce')

listing_with_rates['missing_coord_flag'] = listing_with_rates['Latitude'].isna() | listing_with_rates['Longitude'].isna()

listing_with_rates['zero_coord_flag'] = (
    (listing_with_rates['Latitude'] == 0) |
    (listing_with_rates['Longitude'] == 0)
)

#California coordinates should be negative
listing_with_rates['positive_longitude_flag'] = listing_with_rates['Longitude'] > 0

listing_with_rates['out_of_state_flag'] = (
    (listing_with_rates['Latitude'] < 30) | (listing_with_rates['Latitude'] > 42.5) |
    (listing_with_rates['Longitude'] < -124.5) | (listing_with_rates['Longitude'] > -114)
)

listing_with_rates['implausible_coord_flag'] = (
    (listing_with_rates['Latitude'] < -90) | (listing_with_rates['Latitude'] > 90) |
    (listing_with_rates['Longitude'] < -180) | (listing_with_rates['Longitude'] > 180)
)


print("Missing coordinates:", listing_with_rates['missing_coord_flag'].sum())
print("Zero coordinates:", listing_with_rates['zero_coord_flag'].sum())
print("Positive longitude:", listing_with_rates['positive_longitude_flag'].sum())
print("Out of state coordinates:", listing_with_rates['out_of_state_flag'].sum())
print("Implausible coordinates:", listing_with_rates['implausible_coord_flag'].sum())

Missing coordinates: 111913
Zero coordinates: 94
Positive longitude: 163
Out of state coordinates: 481
Implausible coordinates: 9


### Step 7: Review flag counts

In [33]:
print("Invalid numeric flags:")
print("ClosePrice <= 0:", sold_with_rates['invalid_closeprice_flag'].sum())
print("LivingArea <= 0:", sold_with_rates['invalid_livingarea_flag'].sum())
print("DaysOnMarket < 0:", sold_with_rates['invalid_dom_flag'].sum())
print("BedroomsTotal < 0:", sold_with_rates['invalid_bedrooms_flag'].sum())
print("BathroomsTotalInteger < 0:", sold_with_rates['invalid_bathrooms_flag'].sum())

print("\nDate consistency flags:")
print("Listing after close:", sold_with_rates['listing_after_close_flag'].sum())
print("Purchase after close:", sold_with_rates['purchase_after_close_flag'].sum())
print("Negative timeline:", sold_with_rates['negative_timeline_flag'].sum())

print("\nGeographic data quality flags:")
print("Missing coordinates:", sold_with_rates['missing_coord_flag'].sum())
print("Zero coordinates:", sold_with_rates['zero_coord_flag'].sum())
print("Positive longitude:", sold_with_rates['positive_longitude_flag'].sum())
print("Implausible coordinates:", sold_with_rates['implausible_coord_flag'].sum())

Invalid numeric flags:
ClosePrice <= 0: 35
LivingArea <= 0: 485
DaysOnMarket < 0: 60
BedroomsTotal < 0: 0
BathroomsTotalInteger < 0: 0

Date consistency flags:
Listing after close: 109
Purchase after close: 356
Negative timeline: 379

Geographic data quality flags:
Missing coordinates: 19030
Zero coordinates: 43
Positive longitude: 62
Implausible coordinates: 177


In [34]:
print("Invalid numeric flags:")
print("ClosePrice <= 0:", listing_with_rates['invalid_closeprice_flag'].sum())
print("LivingArea <= 0:", listing_with_rates['invalid_livingarea_flag'].sum())
print("DaysOnMarket < 0:", listing_with_rates['invalid_dom_flag'].sum())
print("BedroomsTotal < 0:", listing_with_rates['invalid_bedrooms_flag'].sum())
print("BathroomsTotalInteger < 0:", listing_with_rates['invalid_bathrooms_flag'].sum())

print("\nDate consistency flags:")
print("Listing after close:", listing_with_rates['listing_after_close_flag'].sum())
print("Purchase after close:", listing_with_rates['purchase_after_close_flag'].sum())
print("Negative timeline:", listing_with_rates['negative_timeline_flag'].sum())

print("\nGeographic data quality flags:")
print("Missing coordinates:", listing_with_rates['missing_coord_flag'].sum())
print("Zero coordinates:", listing_with_rates['zero_coord_flag'].sum())
print("Positive longitude:", listing_with_rates['positive_longitude_flag'].sum())
print("Implausible coordinates:", listing_with_rates['implausible_coord_flag'].sum())

Invalid numeric flags:
ClosePrice <= 0: 12
LivingArea <= 0: 842
DaysOnMarket < 0: 37
BedroomsTotal < 0: 0
BathroomsTotalInteger < 0: 0

Date consistency flags:
Listing after close: 135
Purchase after close: 351
Negative timeline: 397

Geographic data quality flags:
Missing coordinates: 111913
Zero coordinates: 94
Positive longitude: 163
Implausible coordinates: 481


In [35]:
sold_with_rates.to_csv("sold_with_rates_week4-5.csv", index=False)
listing_with_rates.to_csv("listing_with_rates_week4-5.csv", index=False)

In [42]:
print("Shape before cleaning:", sold_with_mortgage_rates.shape)
print("Shape after cleaning:", sold_with_rates.shape)

Shape before cleaning: (591733, 86)
Shape after cleaning: (591733, 84)


In [43]:
print("Shape before cleaning:", listings_with_mortgage_rates.shape)
print("Shape after cleaning:", listing_with_rates.shape)

Shape before cleaning: (852963, 86)
Shape after cleaning: (852963, 74)
